# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`

This notebook provides a step-by-step template for loading and exploring the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via the Croissant schema URL:

[https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata object directly
metadata = dataset.metadata

print(f"Dataset Title: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Identifier: {metadata.identifier}")
print(f"Version: {metadata.version}")
print(f"License: {metadata.license}")

## 2. Data Overview

Review available record sets, fields, and their IDs.

Entities in Croissant (record sets, fields, columns, distributions, etc.) are referenced by their `@id`.


In [ ]:
# List all available record sets and their @id
record_sets = dataset.record_sets
print("Available Record Sets:")
for rs in record_sets:
    print(f"@id: {rs['@id']} | Name: {rs['name']}")

# For each record set, list its fields (column @id and name)
for rs in record_sets:
    print(f"\nFields for Record Set '@id': {rs['@id']} (Name: {rs['name']})")
    for field in rs.get('fields', []):
        print(f"  Field @id: {field['@id']} | Name: {field['name']} | DataType: {field.get('dataType', 'Unknown')}")

## 3. Data Extraction

Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s identified above.


In [ ]:
# Collect all record set @ids
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

# Load all record sets into pandas DataFrames
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded DataFrame for Record Set '{record_set_id}' with shape: {df.shape}")

# Select one record set to preview its columns and first rows
main_rs_id = record_set_ids[0] if len(record_set_ids) > 0 else None
if main_rs_id:
    print("Columns:", dataframes[main_rs_id].columns.tolist())
    display(dataframes[main_rs_id].head())

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes using field/column `@id`s.

In [ ]:
# Example: Analyze the GAD-7 score field (referenced by its @id)
# Find possible numeric fields by inspecting the columns
if main_rs_id:
    df = dataframes[main_rs_id]
    numeric_candidates = [col for col in df.columns if 'gad7' in col.lower() or 'phq9' in col.lower() or 'psq' in col.lower() or df[col].dtype in [int, float]]
    print("Numeric Field Candidates:", numeric_candidates)

    # Let's pick a likely GAD-7 score column (change as needed based on column names)
    numeric_field = numeric_candidates[0] if len(numeric_candidates) > 0 else None

    # Set a threshold for filtering (e.g., scores above 10)
    threshold = 10
    if numeric_field:
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with '{numeric_field}' > {threshold}:")
        display(filtered_df.head())

        # Normalize the numeric field
        norm_col = f"{numeric_field}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized '{numeric_field}' for filtered records:")
        display(filtered_df[[numeric_field, norm_col]].head())

        # Group by a categorical field, such as 'gender' or similar
        group_candidates = [col for col in df.columns if 'gender' in col.lower() or 'education' in col.lower() or df[col].dtype == object]
        group_field = group_candidates[0] if len(group_candidates) > 0 else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped mean '{numeric_field}' by '{group_field}':")
            display(grouped_df.head())

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset using matplotlib or seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_rs_id and numeric_field:
    df = dataframes[main_rs_id]
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field].dropna(), bins=15, kde=True, color='skyblue')
    plt.title(f"Distribution of '{numeric_field}'")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # If we found a group_field, show boxplot
    if group_field:
        plt.figure(figsize=(8, 5))
        sns.boxplot(x=df[group_field], y=df[numeric_field])
        plt.title(f"'{numeric_field}' scores grouped by '{group_field}'")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()

## 6. Conclusion

The FAIR² Mental Health Survey Dataset from Kilifi County, Kenya provides rich insights into mental health indicators across demographics. This exploration demonstrated loading, overview, field extraction using `@id`, filtering by thresholds, normalization, grouping, and visualization. Future analysis could involve deeper statistical modeling and integration across multiple record sets or columns, continually referencing entities by their `@id` for reproducible workflows.
